# numpy08. 文件读写、常用方法速查与综合案例

本章重点：

- NumPy 文件读写
- 文本文件读取：`loadtxt`、`genfromtxt`
- 二进制数组文件：`save`、`load`
- 压缩保存：`savez`
- 常用方法速查表
- 成绩数据综合案例
- 本章练习题


## 1. 文本文件读取

NumPy 可以读取纯数字文本文件。对于混合文本和数字的数据，可以使用 `genfromtxt`。


In [ ]:
import numpy as np
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('numpy_learn/res').exists():
    RES_DIR = Path('numpy_learn/res')

matrix_file = RES_DIR / 'matrix.txt'

# loadtxt 适合读取规则的纯数字文本。
matrix = np.loadtxt(matrix_file)
print(matrix)
print(matrix.dtype)

scores_file = RES_DIR / 'scores.csv'

# genfromtxt 可以处理表头、分隔符、字段名。
# dtype=None 表示让 NumPy 自动推断类型。
# encoding='utf-8' 用于正常读取文本字段。
scores = np.genfromtxt(
    scores_file,
    delimiter=',',
    names=True,
    dtype=None,
    encoding='utf-8'
)

print(scores)
print(scores.dtype)
print(scores['name'])
print(scores['python'])


### 解释

- `np.loadtxt` 更适合纯数字、格式规整的数据。
- `np.genfromtxt` 更灵活，可以读取带表头、缺失值、混合类型的数据。
- `names=True` 表示第一行作为字段名。
- 结构化数组可以通过字段名访问列，例如 `scores['python']`。


## 2. 保存和读取 `.npy`、`.npz`

`.npy` 是 NumPy 原生数组文件格式，适合保存单个数组；`.npz` 可以保存多个数组。


In [ ]:
import numpy as np
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('numpy_learn/res').exists():
    RES_DIR = Path('numpy_learn/res')

WORK_DIR = RES_DIR / 'numpy_workspace'
WORK_DIR.mkdir(exist_ok=True)

arr = np.arange(1, 13).reshape(3, 4)

npy_file = WORK_DIR / 'array_demo.npy'

# save 保存单个数组。
np.save(npy_file, arr)

# load 读取 .npy 文件。
loaded = np.load(npy_file)
print(loaded)

names = np.array(['Tom', 'Jerry', 'Alice'])
scores = np.array([86, 92, 79])

npz_file = WORK_DIR / 'student_demo.npz'

# savez 可以在一个文件中保存多个数组。
np.savez(npz_file, names=names, scores=scores)

data = np.load(npz_file)
print(data.files)
print(data['names'])
print(data['scores'])


### 解释

- `.npy` 能保留数组形状和 dtype。
- `.npz` 像一个数组压缩包，可以保存多个命名数组。
- 读取 `.npz` 后，通过 `data['names']` 访问对应数组。


## 3. 常用方法速查表

| 分类 | 方法 | 说明 |
| --- | --- | --- |
| 创建 | `array`、`arange`、`linspace`、`zeros`、`ones`、`full`、`eye` | 创建数组 |
| 属性 | `shape`、`ndim`、`size`、`dtype` | 查看数组信息 |
| 形状 | `reshape`、`ravel`、`flatten`、`T`、`squeeze`、`expand_dims` | 改变或查看形状 |
| 拼接 | `concatenate`、`vstack`、`hstack` | 合并数组 |
| 拆分 | `split`、`hsplit`、`vsplit` | 拆分数组 |
| 数学 | `abs`、`sqrt`、`exp`、`log`、`round`、`floor`、`ceil` | 逐元素计算 |
| 统计 | `sum`、`mean`、`max`、`min`、`std`、`var`、`argmax`、`argmin` | 聚合统计 |
| 缺失值 | `isnan`、`isinf`、`isfinite`、`nanmean`、`nan_to_num` | 处理 NaN 和无穷 |
| 条件 | 布尔索引、`where`、`isin` | 筛选和条件替换 |
| 排序 | `sort`、`argsort` | 排序和排序索引 |
| 集合 | `unique`、`intersect1d`、`union1d`、`setdiff1d` | 去重和集合操作 |
| 随机 | `default_rng`、`random`、`integers`、`normal`、`choice` | 随机数和抽样 |
| 文件 | `loadtxt`、`genfromtxt`、`save`、`load`、`savez` | 文件读写 |


## 4. 综合案例：成绩数据分析

下面用 NumPy 完成一个小型成绩分析流程：读取 CSV、提取分数、计算统计指标、筛选优秀学生、保存结果。


In [ ]:
import numpy as np
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('numpy_learn/res').exists():
    RES_DIR = Path('numpy_learn/res')

scores_file = RES_DIR / 'scores.csv'
WORK_DIR = RES_DIR / 'numpy_workspace'
WORK_DIR.mkdir(exist_ok=True)

data = np.genfromtxt(
    scores_file,
    delimiter=',',
    names=True,
    dtype=None,
    encoding='utf-8'
)

names = data['name']

# 把三门课成绩拼成二维数组。
# 每一行表示一个学生，每一列表示一门课。
score_matrix = np.vstack([
    data['math'],
    data['english'],
    data['python']
]).T

print('姓名：', names)
print('成绩矩阵：')
print(score_matrix)

# 每个学生平均分。
student_avg = score_matrix.mean(axis=1)
print('学生平均分：', student_avg)

# 每门课平均分。
subject_avg = score_matrix.mean(axis=0)
print('科目平均分：', subject_avg)

# 找出平均分最高的学生。
best_index = np.argmax(student_avg)
print('平均分最高学生：', names[best_index], student_avg[best_index])

# 筛选平均分 >= 85 的学生。
excellent_mask = student_avg >= 85
excellent_names = names[excellent_mask]
excellent_avg = student_avg[excellent_mask]
print('优秀学生：', excellent_names)
print('优秀学生平均分：', excellent_avg)

# 保存优秀学生结果。
result = np.column_stack([excellent_names, excellent_avg.astype(str)])
out_file = WORK_DIR / 'excellent_students_numpy.csv'
np.savetxt(out_file, result, fmt='%s', delimiter=',', header='name,avg', comments='', encoding='utf-8')

print(out_file.read_text(encoding='utf-8'))


### 解释

- `genfromtxt` 读取带姓名和成绩的 CSV。
- `vstack([...]).T` 把三列成绩组合成“学生 x 科目”的矩阵。
- `mean(axis=1)` 计算每个学生平均分。
- `mean(axis=0)` 计算每门课平均分。
- `column_stack` 可以按列合并多个一维数组。
- `savetxt` 可以把结果保存成文本或 CSV。


## 5. 本章练习题

1. 使用 `np.loadtxt` 读取 `matrix.txt`。
2. 保存一个数组到 `.npy` 文件，再读取回来。
3. 使用 `genfromtxt` 读取 `scores.csv`，打印 Python 成绩。
4. 计算每个学生总分。
5. 保存总分最高的学生姓名和总分到 CSV。


In [ ]:
import numpy as np
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('numpy_learn/res').exists():
    RES_DIR = Path('numpy_learn/res')

WORK_DIR = RES_DIR / 'numpy_workspace'
WORK_DIR.mkdir(exist_ok=True)

# 1. 读取 matrix.txt
matrix = np.loadtxt(RES_DIR / 'matrix.txt')
print(matrix)

# 2. 保存和读取 .npy
arr = np.arange(5)
npy_file = WORK_DIR / 'practice.npy'
np.save(npy_file, arr)
print(np.load(npy_file))

# 3. 读取 CSV，打印 Python 成绩
data = np.genfromtxt(
    RES_DIR / 'scores.csv',
    delimiter=',',
    names=True,
    dtype=None,
    encoding='utf-8'
)
print(data['python'])

# 4. 每个学生总分
total = data['math'] + data['english'] + data['python']
print(total)

# 5. 保存总分最高学生
best_index = np.argmax(total)
result = np.array([[data['name'][best_index], str(total[best_index])]])
out_file = WORK_DIR / 'best_student.csv'
np.savetxt(out_file, result, fmt='%s', delimiter=',', header='name,total', comments='', encoding='utf-8')
print(out_file.read_text(encoding='utf-8'))


## 6. 常见错误总结

1. 用 `loadtxt` 读取含文本列的 CSV，导致转换失败。
2. 忘记设置 `delimiter=','`。
3. `genfromtxt` 读取中文或文本时忘记设置 `encoding`。
4. 保存 CSV 时忘记设置 `fmt`，导致字符串保存异常。
5. `.npz` 读取后忘记通过字段名访问数组。
